# **RAG with llamaindex**

### Setting up Enviroment

In [3]:
%pip install llama-index
%pip install llama-index-vector-stores-qdrant
%pip install llama-index-readers-file
%pip install llama-index-embeddings-fastembed
%pip install llama-index-llms-openai
%pip install llama-index-llms-groq
%pip install -U qdrant_client fastembed
%pip install python-dotenv
%pip install gradio

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# Standard library imports
import logging
import sys
import os

# Third-party imports
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Qdrant client import
import qdrant_client

# LlamaIndex core imports
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core import Settings

# LlamaIndex vector store import
from llama_index.vector_stores.qdrant import QdrantVectorStore

# Embedding model imports
from llama_index.embeddings.openai import OpenAIEmbedding

# LLM import
from llama_index.llms.openai import OpenAI
from llama_index.llms.groq import Groq
# Load environment variables
load_dotenv()

# Get OpenAI API key from environment variables
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Setting up Base LLM
Settings.llm = OpenAI(
    model="gpt-4o-mini", temperature=0.1, max_tokens=1024, streaming=True
)

# Set the embedding model
# Option 1: Use FastEmbed with BAAI/bge-base-en-v1.5 model (default)
Settings.embed_model = OpenAIEmbedding(embed_batch_size=10, api_key=OPENAI_API_KEY)

# Qdrant configuration (commented out)
# If you're using Qdrant, uncomment and set these variables:

QDRANT_CLOUD_ENDPOINT=os.getenv("QDRANT_ENDPOINT") 
QDRANT_API_KEY=os.getenv("QDRANT_API_KEY")


### Loading Data

In [2]:
# lets loading the documents using SimpleDirectoryReader

print("🔃 Loading Data")

from llama_index.core import Document
reader = SimpleDirectoryReader("kb_files", recursive=True)
documents = reader.load_data(show_progress=True)

🔃 Loading Data


Loading files: 100%|██████████| 1/1 [00:08<00:00,  8.31s/file]


### Setting up DB

In [4]:
# creating a qdrant client instance

client = qdrant_client.QdrantClient(
    # Qdrant instance address with:
    url=os.getenv("QDRANT_ENDPOINT"),
    # set API KEY for Qdrant Cloud
    api_key=os.getenv("QDRANT_API_KEY")
)

vector_store = QdrantVectorStore(client=client, collection_name="01_Basic_RAG")

In [5]:
## ingesting data into vector database

## lets set up an ingestion pipeline

from llama_index.core.node_parser import TokenTextSplitter
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(
    transformations=[
        # MarkdownNodeParser(include_metadata=True),
        # TokenTextSplitter(chunk_size=500, chunk_overlap=20),
        SentenceSplitter(chunk_size=1024, chunk_overlap=20),
        # SemanticSplitterNodeParser(buffer_size=1, breakpoint_percentile_threshold=95 , embed_model=Settings.embed_model),
        Settings.embed_model,
    ],
    vector_store=vector_store,
)

# Ingest directly into a vector db
nodes = pipeline.run(documents=documents , show_progress=True)
print("Number of chunks added to vector DB :",len(nodes))

Parsing nodes:   0%|          | 0/357 [00:00<?, ?it/s]

Generating embeddings:   0%|          | 0/435 [00:00<?, ?it/s]

Number of chunks added to vector DB : 435


### Setting up Index

In [6]:
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

### Prompt Engineering

In [7]:
from llama_index.core import ChatPromptTemplate

qa_prompt_str = (
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, "
    "answer the question: {query_str}\n"
)

refine_prompt_str = (
    "We have the opportunity to refine the original answer "
    "(only if needed) with some more context below.\n"
    "------------\n"
    "{context_msg}\n"
    "------------\n"
    "Given the new context, refine the original answer to better "
    "answer the question: {query_str}. "
    "If the context isn't useful, output the original answer again.\n"
    "Original Answer: {existing_answer}"
)

# Text QA Prompt
chat_text_qa_msgs = [
    ("system","You are a AI assistant who is well versed with answering questions from the provided context"),
    ("user", qa_prompt_str),
]
text_qa_template = ChatPromptTemplate.from_messages(chat_text_qa_msgs)

# Refine Prompt
chat_refine_msgs = [
    ("system","Always answer the question, even if the context isn't helpful.",),
    ("user", refine_prompt_str),
]
refine_template = ChatPromptTemplate.from_messages(chat_refine_msgs)

In [8]:
chat_query = "What is in this document"

# Setting up Chat Engine
BASE_RAG_CHAT_ENGINE = index.as_chat_engine()

response = BASE_RAG_CHAT_ENGINE.chat(chat_query)
display(Markdown(str(response)))

Could you please provide the document or specify its contents so I can assist you better?

### Simple App

In [9]:
from typing import List
from llama_index.core.base.llms.types import ChatMessage, MessageRole

class ChatEngineInterface:
    def __init__(self, index):
        self.chat_engine = index.as_chat_engine()
        self.chat_history: List[ChatMessage] = []

    def display_message(self, role: str, content: str):
        if role == "USER":
            display(Markdown(f"**Human:** {content}"))
        else:
            display(Markdown(f"**AI:** {content}"))

    def chat(self, message: str) -> str:
        # Create a ChatMessage for the user input
        user_message = ChatMessage(role=MessageRole.USER, content=message)
        self.chat_history.append(user_message)

        # Get response from the chat engine
        response = self.chat_engine.chat(message, chat_history=self.chat_history)

        # Create a ChatMessage for the AI response
        ai_message = ChatMessage(role=MessageRole.ASSISTANT, content=str(response))
        self.chat_history.append(ai_message)

        # Display the conversation
        self.display_message("USER", message)
        self.display_message("ASSISTANT", str(response))

        print("\n" + "-"*50 + "\n")  # Separator for readability

        return str(response)

    def get_chat_history(self) -> List[ChatMessage]:
        return self.chat_history

In [10]:
chat_interface = ChatEngineInterface(index)
while True:
    user_input = input("You: ").strip()
    if user_input.lower() == 'exit':
        print("Thank you for chatting! Goodbye.")
        break
    chat_interface.chat(user_input)

**Human:** How many articles are in this document

**AI:** Could you please provide the document or specify the content you are referring to?


--------------------------------------------------



**Human:** what is la ley general de la salud about

**AI:** La Ley General de Salud regula el derecho a la protección de la salud de las personas en México. Establece las bases y modalidades para el acceso a los servicios de salud, distribuye competencias entre la Federación y las entidades federativas en materia de salubridad general, y busca promover el bienestar físico y mental, mejorar la calidad de vida, y fomentar actitudes solidarias en la preservación y mejora de la salud.

...

**AI:** Los artículos más relevantes de La Ley General de Salud para farmacias incluyen:

- **Artículo 258**: Establece que las farmacias deben contar con la licencia sanitaria correspondiente expedida por la Secretaría de Salud y cumplir con las normativas de la Farmacopea de los Estados Unidos Mexicanos.

- **Artículo 259**: Indica que las farmacias deben tener un responsable de la identidad, pureza y seguridad de los productos, quien debe cumplir con los requisitos establecidos por las disposiciones aplicables.

- **Artículo 226**: Define las categorías de medicamentos que pueden ser vendidos en farmacias, incluyendo aquellos que requieren receta médica y los que no, así como las condiciones para su adquisición y prescripción.

Estos artículos regulan la operación, la responsabilidad y la clasificación de los medicamentos en las farmacias.


--------------------------------------------------



**Human:** 

**AI:** ¿Hay algo más en lo que pueda ayudarte?


--------------------------------------------------

Thank you for chatting! Goodbye.


In [11]:
# To view chat history:
history = chat_interface.get_chat_history()
for message in history:
    print(f"{message.role}: {message.content}")

MessageRole.USER: How many articles are in this document
MessageRole.USER: How many articles are in this document
MessageRole.ASSISTANT: Could you please provide the document or specify the content you are referring to?
MessageRole.ASSISTANT: Could you please provide the document or specify the content you are referring to?
MessageRole.USER: what is la ley general de la salud about
MessageRole.USER: what is la ley general de la salud about
MessageRole.ASSISTANT: None
MessageRole.TOOL: La Ley General de Salud regula el derecho a la protección de la salud de las personas en México, estableciendo las bases y modalidades para el acceso a los servicios de salud. También distribuye competencias entre la Federación y las entidades federativas en materia de salubridad general, y busca promover el bienestar físico y mental, mejorar la calidad de vida, y fomentar actitudes solidarias en la preservación y mejora de la salud.
MessageRole.ASSISTANT: La Ley General de Salud regula el derecho a la pr

### Gradio App

In [ ]:
import gradio as gr
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Document, Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
import qdrant_client
import os
import tempfile
import shutil
from typing import List
from llama_index.core.base.llms.types import ChatMessage, MessageRole

class RAGChatbot:
    def __init__(self):
        self.client = qdrant_client.QdrantClient(path="./Demo_RAG")
        self.vector_store = None
        self.index = None
        self.chat_engine = None
        self.chat_history = []
        # Initialize vector store and index
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name="Demo_RAG"
        )

        # Create the index and ingest documents
        self.index = VectorStoreIndex.from_vector_store(
            vector_store=self.vector_store
        )

        # Initialize chat engine
        self.chat_engine = self.index.as_chat_engine(
            streaming=True,
            verbose=True
        )


    def process_uploaded_files(self, files) -> str:
        try:
            # Create a temporary directory for processing
            with tempfile.TemporaryDirectory() as temp_dir:
                # Save uploaded files to temporary directory
                for file in files:
                    shutil.copy(file.name, temp_dir)

                # Load documents
                reader = SimpleDirectoryReader(temp_dir)
                documents = reader.load_data()

                pipeline = IngestionPipeline(
                    transformations=[
                        # MarkdownNodeParser(include_metadata=True),
                        # TokenTextSplitter(chunk_size=500, chunk_overlap=20),
                        SentenceSplitter(chunk_size=1024, chunk_overlap=20),
                        # SemanticSplitterNodeParser(buffer_size=1, breakpoint_percentile_threshold=95 , embed_model=Settings.embed_model),
                        Settings.embed_model,
                    ],
                    vector_store=self.vector_store,
                )

                # Ingest directly into a vector db
                nodes = pipeline.run(documents=documents , show_progress=True)

                return f"Successfully processed {len(documents)} documents. Ready to chat! and inserted {len(nodes)} into the database"

        except Exception as e:
            return f"Error processing files: {str(e)}"

    def chat(self, message: str, history: List[List[str]]) -> List[List[str]]:
        if self.chat_engine is None:
            return history + [[message, "Please upload documents first before starting the chat."]]

        try:
            # Convert history to ChatMessage format
            chat_history = []
            for h in history:
                chat_history.extend([
                    ChatMessage(role=MessageRole.USER, content=h[0]),
                    ChatMessage(role=MessageRole.ASSISTANT, content=h[1])
                ])

            # Add current message to history
            chat_history.append(ChatMessage(role=MessageRole.USER, content=message))

            # Get response from chat engine
            response = self.chat_engine.chat(message, chat_history=chat_history)

            # Return the updated history with the new message pair
            return history + [[message, str(response)]]

        except Exception as e:
            return history + [[message, f"Error generating response: {str(e)}"]]

def create_demo():
    # Initialize the chatbot
    chatbot = RAGChatbot()

    with gr.Blocks(theme=gr.themes.Soft()) as demo:
        gr.Markdown("# RAG Chatbot")
        gr.Markdown("Upload your documents and start chatting!")

        with gr.Row():
            with gr.Column(scale=1):
                file_output = gr.File(
                    file_count="multiple",
                    label="Upload Documents",
                    file_types=[".txt", ".pdf", ".docx", ".md"]
                )
                upload_button = gr.Button("Process Documents")
                status_box = gr.Textbox(label="Status", interactive=False)

            with gr.Column(scale=2):
                chatbot_interface = gr.Chatbot(
                    label="Chat History",
                    height=400,
                    bubble_full_width=False,
                )
                with gr.Row():
                    msg = gr.Textbox(
                        label="Type your message",
                        placeholder="Ask me anything about the uploaded documents...",
                        lines=2,
                        scale=4
                    )
                    submit_button = gr.Button("Submit", scale=1)
                clear = gr.Button("Clear")

        # Event handlers
        upload_button.click(
            fn=chatbot.process_uploaded_files,
            inputs=[file_output],
            outputs=[status_box],
        )
        submit_button.click(
            fn=chatbot.chat,
            inputs=[msg, chatbot_interface],
            outputs=[chatbot_interface],
        )

        clear.click(
            lambda: None,
            None,
            chatbot_interface,
            queue=False
        )

    return demo

if __name__ == "__main__":
    demo = create_demo()
    demo.launch(share=True)